In [2]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv

d:\Computer Courses\Agenetic-Ai\langchain_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()

model = ChatGroq(
    model = 'openai/gpt-oss-20b',
    temperature = 0
)

In [4]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str
    evaluate: float

In [5]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state

In [6]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the following outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [20]:
def evaluate(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']
    content = state['content']

    prompt = f'''
    You are a professional blog evaluator.
    Evaluate the following blog based on:

    1. Title quality
    2. Content quality
    3. Structure and organization
    4. Clarity and readability
    5. Engagement
    6. Grammer
    7. Overall usefulness

    Give:
    - Overall score out of 10
    - Score for each criterion
    - Strenghts
    - Weakness

    Title: {title}
    Outline: {outline}
    content: {content}
    '''

    evaluate = model.invoke(prompt).content

    state['evaluate'] = evaluate

    return state

In [21]:
graph = StateGraph(BlogState)

# add nodes
graph.add_node('create_outline', create_outline),
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate', evaluate)

# add edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate')
graph.add_edge('evaluate', END)

workflow = graph.compile()

In [22]:
initial_state = {'title':'Rise of AI in India'}

final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'Rise of AI in India', 'outline': '## Blog Outline  \n**Title:** *Rise of AI in India: From Start‑ups to Smart Nation*\n\n---\n\n### 1. Hook & Thesis  \n- **Opening anecdote / statistic** – e.g., “India’s AI market is projected to hit $12\u202fbillion by 2025.”  \n- **Thesis statement** – India is rapidly becoming a global AI powerhouse, driven by a vibrant ecosystem, supportive policy, and a massive talent pool.\n\n---\n\n### 2. Historical Context  \n| Section | Key Points |\n|---------|------------|\n| **Early Beginnings** | • 1990s research labs (IISc, IITs) – first AI projects. |\n| **Government Vision** | • 2008 “National Knowledge Network” – early digital infrastructure. |\n| **First AI Start‑ups** | • 2010‑2013: AIB, Haptik, and other early players. |\n\n---\n\n### 3. Current Landscape  \n#### 3.1 Market Size & Growth  \n- CAGR 2023‑2028: 30%+  \n- Segments: Healthcare, FinTech, Agriculture, Retail, Smart Cities.\n\n#### 3.2 Key Players  \n- **Large Corporations** – In

In [23]:
print(final_state['outline'])

## Blog Outline  
**Title:** *Rise of AI in India: From Start‑ups to Smart Nation*

---

### 1. Hook & Thesis  
- **Opening anecdote / statistic** – e.g., “India’s AI market is projected to hit $12 billion by 2025.”  
- **Thesis statement** – India is rapidly becoming a global AI powerhouse, driven by a vibrant ecosystem, supportive policy, and a massive talent pool.

---

### 2. Historical Context  
| Section | Key Points |
|---------|------------|
| **Early Beginnings** | • 1990s research labs (IISc, IITs) – first AI projects. |
| **Government Vision** | • 2008 “National Knowledge Network” – early digital infrastructure. |
| **First AI Start‑ups** | • 2010‑2013: AIB, Haptik, and other early players. |

---

### 3. Current Landscape  
#### 3.1 Market Size & Growth  
- CAGR 2023‑2028: 30%+  
- Segments: Healthcare, FinTech, Agriculture, Retail, Smart Cities.

#### 3.2 Key Players  
- **Large Corporations** – Infosys Nia, Wipro HOLMES, TCS Ignio.  
- **Start‑ups** – Haptik, Niki.ai, Sig

In [24]:
print(final_state['content'])

# Rise of AI in India: From Start‑ups to Smart Nation  

---

## 1. Hook & Thesis  

> **“India’s AI market is projected to hit $12 billion by 2025.”**  
> That figure is not just a headline; it’s a snapshot of a nation that is turning its youthful energy, massive data streams, and a growing digital economy into a global AI powerhouse.  

**Thesis:** India is rapidly becoming a global AI powerhouse, driven by a vibrant ecosystem of start‑ups, a supportive policy framework, and an unprecedented talent pool. From the first research labs in the 1990s to today’s AI‑powered smart cities, the country is rewriting the rules of innovation.

---

## 2. Historical Context  

| Section | Key Points |
|---------|------------|
| **Early Beginnings** | • 1990s research labs (IISc, IITs) – first AI projects.<br>• Early focus on symbolic AI, expert systems, and rule‑based engines. |
| **Government Vision** | • 2008 “National Knowledge Network” – laid the groundwork for high‑speed connectivity.<br>• 20

In [25]:
print(final_state['evaluate'])

**Overall Score:** **8.5 / 10**

| Criterion | Score (out of 10) | Comments |
|-----------|-------------------|----------|
| 1. Title quality | **8** | “Rise of AI in India: From Start‑ups to Smart Nation” is descriptive, keyword‑rich, and signals scope. Could be a bit punchier, but it works well. |
| 2. Content quality | **9** | The post is data‑rich, covers history, market, policy, sector stories, challenges, academia, and future outlook. It reads like a well‑researched white‑paper. |
| 3. Structure & organization | **9** | Logical flow, clear headings, tables, bullet lists, and a call‑to‑action. The outline and final draft align perfectly. |
| 4. Clarity & readability | **8** | Mostly clear prose; some dense paragraphs could be broken up. The use of tables and bullet points improves readability. |
| 5. Engagement | **8** | Strong hook, real‑world examples, and a CTA keep readers interested. Visual suggestions add an interactive layer. |
| 6. Grammar | **9** | Minor typographical sli